# Counting CNN (Convolutional Neural Network) parameters

_Deep Learning_

**How many weights a convolutional layer holds, and what a neuron can 'see'.**

A convolutional layer learns the numbers inside its filters. Counting them tells you the layer's size.

     Each filter is $F\times F$ wide and goes through all $C$ input channels (like red, green, blue). Plus one bias each.

---

This notebook builds up the parameter count one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We build a small two-layer convolutional network and count its weights two ways: by the hand formula and by asking PyTorch directly. We do it in three steps: (1) define the layers, (2) count layer 1 by hand, (3) confirm against PyTorch's own count.

### Step 1 — Define the two convolutional layers

We stack two `Conv2d` layers (each followed by a ReLU). The first maps a 3-channel input (like an RGB image) to 16 feature maps; the second maps those 16 maps to 32. Every filter is 3x3, and `padding=1` keeps the spatial size unchanged. The numbers that matter for counting are the channel counts and the 3x3 filter size.

In [ ]:
import torch
import torch.nn as nn

cnn = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),    # layer 1: 3 input channels -> 16 feature maps
    nn.ReLU(),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),   # layer 2: 16 -> 32 feature maps
    nn.ReLU(),
)

### Step 2 — Count layer 1 by hand

Each filter in layer 1 looks at all 3 input channels through a 3x3 window, so it holds `3 * 3 * 3 = 27` weights. There are 16 such filters, plus one bias per filter. That gives `16 * (3*3*3) + 16 = 448` learnable numbers — the formula `out * (in * fh * fw) + out`.

In [ ]:
out_channels = 16
in_channels = 3
filter_height = 3
filter_width = 3

weights_per_filter = in_channels * filter_height * filter_width   # 3*3*3 = 27
weight_count = out_channels * weights_per_filter                  # 16 * 27 = 432
bias_count = out_channels                                         # one bias per filter
manual = weight_count + bias_count                               # 432 + 16 = 448

print("layer-1 params (by hand):", manual)        # 448

### Step 3 — Confirm against PyTorch's own count

PyTorch tracks every parameter tensor, so summing `numel()` over a layer's parameters gives its exact count. We check each conv layer, then the whole network. Layer 1 should match our hand calculation of 448.

In [ ]:
named_layers = [("conv1", cnn[0]), ("conv2", cnn[2])]

for name, layer in named_layers:
    n = sum(p.numel() for p in layer.parameters())
    print(name, "params:", n)

total = sum(p.numel() for p in cnn.parameters())
print("total:", total)

## Visualize the data & results

_For real 8x8 digit images, why is a convolutional layer so much cheaper than a fully-connected one?_

### Step 1 — Count conv vs fully-connected params on a real image

Using the real 8x8 digit images, we compute three parameter counts with the same `out * (in*fh*fw) + out` rule for the conv layers. The fully-connected (FC) layer instead connects every one of the 64 flattened pixels to all 128 output units, so its weight count explodes to `64*128 + 128`. Notice the conv layers reuse one small filter across the whole image, while the FC layer needs a separate weight per pixel.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
h, w = digits.images[0].shape            # real image = 8 x 8, 1 channel

conv1 = 16 * (1 * 3 * 3) + 16            # 160:  1 channel in -> 16 maps, 3x3
conv2 = 32 * (16 * 3 * 3) + 32           # 4640: 16 maps in -> 32 maps, 3x3
fc = (h * w * 1) * 128 + 128             # 8320: every flattened pixel -> 128 units

print("conv1:", conv1)
print("conv2:", conv2)
print("FC   :", fc)

### Step 2 — Plot the counts on a log scale

We bar-chart the three counts. A log y-axis is used because the FC layer dwarfs the conv layers — even on this tiny 8x8 input the fully-connected layer needs far more weights, which is exactly why convolutions are the cheaper choice for images.

In [ ]:
labels = ["conv1 (1->16, 3x3)", "conv2 (16->32, 3x3)", "FC (64->128)"]
values = [conv1, conv2, fc]
colors = ["#7ee787", "#4ea1ff", "#ff7b72"]

plt.bar(labels, values, color=colors)
plt.title("Parameter count for an 8x8 digit input: conv vs FC")
plt.yscale("log")
plt.xticks(rotation=15)
plt.show()